In [1]:

import sqlite3
import pandas as pd
from pathlib import Path

db_path = Path("stocks.db")

if not db_path.exists():
    raise FileNotFoundError(
        "stocks.db was not found in the current working directory."
    )

conn = sqlite3.connect(db_path)

tables = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
""", conn)

tables

,name
0,fundamentals
1,prices_adjusted
2,prices_unadjusted
3,securities


### Top 20 profitable stocks

In [ ]:

query = """
WITH ranked_prices AS (
    SELECT
        symbol,
        date,hn
        close,
        ROW_NUMBER() OVER (
            PARTITION BY symbol ORDER BY date ASC
        ) AS first_rank,
        ROW_NUMBER() OVER (
            PARTITION BY symbol ORDER BY date DESC
        ) AS last_rank
    FROM prices_adjusted
    WHERE close IS NOT NULL
)
SELECT
    symbol,
    MAX(CASE WHEN first_rank = 1 THEN date END) AS first_date,
    ROUND(MAX(CASE WHEN first_rank = 1 THEN close END), 2)
        AS first_close,
    MAX(CASE WHEN last_rank = 1 THEN date END) AS last_date,
    ROUND(MAX(CASE WHEN last_rank = 1 THEN close END), 2)
        AS last_close,
    ROUND(
        100.0 * (
            MAX(CASE WHEN last_rank = 1 THEN close END) /
            MAX(CASE WHEN first_rank = 1 THEN close END) - 1
        ),
        2
    ) AS price_return_pct
FROM ranked_prices
GROUP BY symbol
HAVING MAX(CASE WHEN first_rank = 1 THEN close END) > 0
ORDER BY price_return_pct DESC
LIMIT 20;
"""

results = pd.read_sql_query(query, conn)

results

,symbol,first_date,first_close,last_date,last_close,price_return_pct
0,NFLX,2010-01-04,7.64,2016-12-30,123.80,1520.42
1,REGN,2010-01-04,24.63,2016-12-30,367.09,1390.42
2,ULTA,2010-01-04,18.85,2016-12-30,254.94,1252.47
3,ALK,2010-01-04,8.72,2016-12-30,88.73,917.84
4,AAL,2010-01-04,4.77,2016-12-30,46.69,878.83
5,STZ,2010-01-04,16.12,2016-12-30,153.31,851.05
6,AVGO,2010-01-04,18.86,2016-12-30,176.77,837.27
7,CHTR,2010-01-05,35.00,2016-12-30,287.92,722.63
8,ORLY,2010-01-04,38.50,2016-12-30,278.41,623.14
9,MNST,2010-01-04,6.55,2016-12-30,44.34,577.29


### Top 20 lowest profitable stocks

In [3]:
query = """
WITH ranked_prices AS (
    SELECT
        symbol,
        date,
        close,
        ROW_NUMBER() OVER (
            PARTITION BY symbol ORDER BY date ASC
        ) AS first_rank,
        ROW_NUMBER() OVER (
            PARTITION BY symbol ORDER BY date DESC
        ) AS last_rank
    FROM prices_adjusted
    WHERE close IS NOT NULL
)
SELECT
    symbol,
    MAX(CASE WHEN first_rank = 1 THEN date END) AS first_date,
    ROUND(MAX(CASE WHEN first_rank = 1 THEN close END), 2)
        AS first_close,
    MAX(CASE WHEN last_rank = 1 THEN date END) AS last_date,
    ROUND(MAX(CASE WHEN last_rank = 1 THEN close END), 2)
        AS last_close,
    ROUND(
        100.0 * (
            MAX(CASE WHEN last_rank = 1 THEN close END) /
            MAX(CASE WHEN first_rank = 1 THEN close END) - 1
        ),
        2
    ) AS price_return_pct
FROM ranked_prices
GROUP BY symbol
HAVING MAX(CASE WHEN first_rank = 1 THEN close END) > 0
ORDER BY price_return_pct ASC
LIMIT 20;
"""

least_profitable = pd.read_sql_query(query, conn)

least_profitable

,symbol,first_date,first_close,last_date,last_close,price_return_pct
0,SWN,2010-01-04,50.83,2016-12-30,10.82,-78.71
1,FSLR,2010-01-04,135.46,2016-12-30,32.09,-76.31
2,CHK,2010-01-04,26.58,2016-12-30,7.02,-73.58
3,FCX,2010-01-04,41.73,2016-12-30,13.19,-68.39
4,SPLS,2010-01-04,24.50,2016-12-30,9.05,-63.06
5,FTR,2010-01-04,7.96,2016-12-30,3.38,-57.54
6,MOS,2010-01-04,61.71,2016-12-30,29.33,-52.47
7,KMI,2013-01-02,36.53,2016-12-30,20.71,-43.31
8,DVN,2010-01-04,76.57,2016-12-30,45.67,-40.36
9,APA,2010-01-04,105.87,2016-12-30,63.47,-40.05


### Stock trading close to all time high

In [4]:
query = """
WITH ranked_prices AS (
    SELECT
        symbol,
        date,
        close,
        MAX(close) OVER (
            PARTITION BY symbol
        ) AS all_time_high,
        ROW_NUMBER() OVER (
            PARTITION BY symbol ORDER BY date DESC
        ) AS latest_rank
    FROM prices_adjusted
    WHERE close IS NOT NULL
)
SELECT
    symbol,
    date AS latest_date,
    ROUND(close, 2) AS latest_close,
    ROUND(all_time_high, 2) AS all_time_high,
    ROUND(
        100.0 * (close / all_time_high - 1), 2
    ) AS pct_from_ath
FROM ranked_prices
WHERE latest_rank = 1
  AND all_time_high > 0
ORDER BY pct_from_ath DESC
LIMIT 20;
"""

near_ath = pd.read_sql_query(query, conn)
near_ath

,symbol,latest_date,latest_close,all_time_high,pct_from_ath
0,TRV,2016-12-30,122.42,122.57,-0.12
1,AJG,2016-12-30,51.96,52.12,-0.31
2,WM,2016-12-30,70.91,71.14,-0.32
3,LLTC,2016-12-30,62.35,62.66,-0.49
4,ALL,2016-12-30,74.12,74.58,-0.62
5,PGR,2016-12-30,35.50,35.74,-0.67
6,CAG,2016-12-30,39.55,39.85,-0.75
7,DGX,2016-12-30,91.90,92.60,-0.76
8,RSG,2016-12-30,57.05,57.50,-0.78
9,DE,2016-12-30,103.04,103.92,-0.85


### Top 20 biggest historical drawdowns
stocks experienced the largest falls from a previous peak

In [5]:
query = """
WITH running_prices AS (
    SELECT
        symbol,
        date,
        close,
        MAX(close) OVER (
            PARTITION BY symbol
            ORDER BY date
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS running_peak
    FROM prices_adjusted
    WHERE close IS NOT NULL
),
drawdowns AS (
    SELECT
        symbol,
        date,
        close,
        running_peak,
        100.0 * (close / running_peak - 1) AS drawdown_pct
    FROM running_prices
    WHERE running_peak > 0
),
ranked_drawdowns AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY symbol
            ORDER BY drawdown_pct ASC, date DESC
        ) AS drawdown_rank
    FROM drawdowns
)
SELECT
    symbol,
    date AS trough_date,
    ROUND(close, 2) AS trough_close,
    ROUND(running_peak, 2) AS previous_peak,
    ROUND(drawdown_pct, 2) AS drawdown_pct
FROM ranked_drawdowns
WHERE drawdown_rank = 1
ORDER BY drawdown_pct ASC
LIMIT 20;
"""

biggest_drawdowns = pd.read_sql_query(query, conn)
biggest_drawdowns

,symbol,trough_date,trough_close,previous_peak,drawdown_pct
0,CHK,2016-02-12,1.59,33.69,-95.28
1,FCX,2016-01-13,3.74,60.92,-93.86
2,FSLR,2012-06-01,11.77,170.80,-93.11
3,SWN,2015-12-17,5.15,51.65,-90.03
4,MRO,2016-02-19,6.73,41.69,-83.86
5,NFLX,2012-09-25,7.69,42.68,-81.99
6,WMB,2016-02-08,11.16,60.86,-81.66
7,DVN,2016-02-19,18.65,93.10,-79.97
8,WYNN,2016-01-14,51.50,247.95,-79.23
9,HPQ,2012-11-20,5.32,24.76,-78.52


### Top 20 most volatile stocks

In [6]:
query = """
WITH daily_returns AS (
    SELECT
        symbol,
        date,
        100.0 * (
            close / LAG(close) OVER (
                PARTITION BY symbol ORDER BY date
            ) - 1
        ) AS daily_return_pct
    FROM prices_adjusted
    WHERE close IS NOT NULL
),
volatility AS (
    SELECT
        symbol,
        COUNT(daily_return_pct) AS trading_days,
        AVG(ABS(daily_return_pct)) AS avg_abs_daily_move_pct,
        MAX(ABS(daily_return_pct)) AS largest_daily_move_pct
    FROM daily_returns
    WHERE daily_return_pct IS NOT NULL
    GROUP BY symbol
)
SELECT
    symbol,
    trading_days,
    ROUND(avg_abs_daily_move_pct, 2) AS avg_daily_move_pct,
    ROUND(largest_daily_move_pct, 2) AS largest_daily_move_pct
FROM volatility
WHERE trading_days >= 100
ORDER BY avg_abs_daily_move_pct DESC
LIMIT 20;
"""

most_volatile = pd.read_sql_query(query, conn)
most_volatile

,symbol,trading_days,avg_daily_move_pct,largest_daily_move_pct
0,FSLR,1761,2.43,45.53
1,CHK,1761,2.38,34.44
2,NFLX,1761,2.25,42.22
3,MU,1761,2.23,23.44
4,AAL,1761,2.22,17.32
5,QRVO,503,2.20,23.23
6,FCX,1761,2.18,28.66
7,UAL,1761,2.04,17.52
8,SWN,1761,2.02,19.24
9,REGN,1761,1.97,28.60


### Stocks with most positive years

In [7]:
query = """
WITH yearly_prices AS (
    SELECT
        symbol,
        strftime('%Y', date) AS year,
        date,
        close,
        ROW_NUMBER() OVER (
            PARTITION BY symbol, strftime('%Y', date)
            ORDER BY date ASC
        ) AS first_rank,
        ROW_NUMBER() OVER (
            PARTITION BY symbol, strftime('%Y', date)
            ORDER BY date DESC
        ) AS last_rank
    FROM prices_adjusted
    WHERE close IS NOT NULL
),
annual_returns AS (
    SELECT
        symbol,
        year,
        MAX(CASE WHEN first_rank = 1 THEN close END) AS first_close,
        MAX(CASE WHEN last_rank = 1 THEN close END) AS last_close,
        100.0 * (
            MAX(CASE WHEN last_rank = 1 THEN close END) /
            MAX(CASE WHEN first_rank = 1 THEN close END) - 1
        ) AS annual_return_pct
    FROM yearly_prices
    GROUP BY symbol, year
    HAVING MAX(CASE WHEN first_rank = 1 THEN close END) > 0
)
SELECT
    symbol,
    COUNT(*) AS years_observed,
    SUM(CASE WHEN annual_return_pct > 0 THEN 1 ELSE 0 END)
        AS positive_years,
    SUM(CASE WHEN annual_return_pct < 0 THEN 1 ELSE 0 END)
        AS negative_years,
    ROUND(
        100.0 * SUM(
            CASE WHEN annual_return_pct > 0 THEN 1 ELSE 0 END
        ) / COUNT(*), 2
    ) AS positive_year_pct,
    ROUND(AVG(annual_return_pct), 2) AS avg_annual_return_pct
FROM annual_returns
GROUP BY symbol
HAVING COUNT(*) >= 3
ORDER BY positive_year_pct DESC, avg_annual_return_pct DESC
LIMIT 20;
"""

consistent_stocks = pd.read_sql_query(query, conn)
consistent_stocks

,symbol,years_observed,positive_years,negative_years,positive_year_pct,avg_annual_return_pct
0,FB,4,4,0,100.0,45.94
1,AVGO,7,7,0,100.0,40.94
2,ALK,7,7,0,100.0,40.82
3,CHTR,7,7,0,100.0,37.65
4,ORLY,7,7,0,100.0,34.42
5,FL,7,7,0,100.0,32.00
6,ROST,7,7,0,100.0,30.20
7,AZO,7,7,0,100.0,28.08
8,UNH,7,7,0,100.0,26.31
9,HD,7,7,0,100.0,25.14


### Top 20 stocks by recent 20 session momentum

Stocks that have gained the most over their latest 20 recorded trading sessions

In [8]:
query = """
WITH ranked_prices AS (
    SELECT
        symbol,
        date,
        close,
        ROW_NUMBER() OVER (
            PARTITION BY symbol ORDER BY date DESC
        ) AS rn
    FROM prices_adjusted
    WHERE close IS NOT NULL
),
recent_returns AS (
    SELECT
        symbol,
        MAX(CASE WHEN rn = 1 THEN date END) AS latest_date,
        MAX(CASE WHEN rn = 1 THEN close END) AS latest_close,
        MAX(CASE WHEN rn = 20 THEN close END) AS close_20_sessions_ago
    FROM ranked_prices
    WHERE rn <= 20
    GROUP BY symbol
)
SELECT
    symbol,
    latest_date,
    ROUND(latest_close, 2) AS latest_close,
    ROUND(close_20_sessions_ago, 2) AS close_20_sessions_ago,
    ROUND(
        100.0 * (
            latest_close / close_20_sessions_ago - 1
        ), 2
    ) AS return_20_sessions_pct
FROM recent_returns
WHERE close_20_sessions_ago > 0
ORDER BY return_20_sessions_pct DESC
LIMIT 20;
"""

recent_momentum = pd.read_sql_query(query, conn)
recent_momentum

,symbol,latest_date,latest_close,close_20_sessions_ago,return_20_sessions_pct
0,NVDA,2016-12-30,106.74,88.45,20.68
1,MU,2016-12-30,21.92,18.79,16.66
2,EW,2016-12-30,93.70,83.35,12.42
3,XLNX,2016-12-30,60.37,54.22,11.34
4,VLO,2016-12-30,68.32,61.46,11.16
5,KMX,2016-12-30,64.39,57.99,11.04
6,CF,2016-12-30,31.48,28.38,10.92
7,DLR,2016-12-30,98.26,89.10,10.28
8,T,2016-12-30,42.53,38.61,10.15
9,EXR,2016-12-30,77.24,70.18,10.06


### Whether momentum predicts future returns

In [9]:
query = """
WITH price_history AS (
    SELECT
        symbol,
        date,
        close,

        LAG(close, 20) OVER (
            PARTITION BY symbol
            ORDER BY date
        ) AS close_20_sessions_ago,

        LEAD(close, 20) OVER (
            PARTITION BY symbol
            ORDER BY date
        ) AS close_20_sessions_later

    FROM prices_adjusted
    WHERE close > 0
),

signals AS (
    SELECT
        symbol,
        date,

        close / close_20_sessions_ago - 1
            AS past_20_return,

        close_20_sessions_later / close - 1
            AS future_20_return

    FROM price_history
    WHERE close_20_sessions_ago > 0
      AND close_20_sessions_later > 0
),

ranked_stocks AS (
    SELECT
        *,
        NTILE(5) OVER (
            PARTITION BY date
            ORDER BY past_20_return
        ) AS momentum_quintile

    FROM signals
),

group_performance AS (
    SELECT
        momentum_quintile,
        COUNT(*) AS observations,

        AVG(past_20_return) * 100
            AS avg_past_return_pct,

        AVG(future_20_return) * 100
            AS avg_future_return_pct,

        100.0 * AVG(
            CASE
                WHEN future_20_return > 0 THEN 1.0
                ELSE 0.0
            END
        ) AS positive_return_pct

    FROM ranked_stocks
    GROUP BY momentum_quintile
)

SELECT
    momentum_quintile,
    observations,
    ROUND(avg_past_return_pct, 2) AS avg_past_return_pct,
    ROUND(avg_future_return_pct, 2) AS avg_future_return_pct,
    ROUND(positive_return_pct, 2) AS positive_return_pct,

    ROUND(
        (SELECT avg_future_return_pct
         FROM group_performance
         WHERE momentum_quintile = 5)
        -
        (SELECT avg_future_return_pct
         FROM group_performance
         WHERE momentum_quintile = 1),
        2
    ) AS top_vs_bottom_spread_pct

FROM group_performance
ORDER BY momentum_quintile;
"""

momentum_research = pd.read_sql_query(query, conn)
momentum_research

,momentum_quintile,observations,avg_past_return_pct,avg_future_return_pct,positive_return_pct,top_vs_bottom_spread_pct
0,1,149112,-7.01,1.28,56.56,-0.11
1,2,149079,-1.66,1.27,58.21,-0.11
2,3,148642,1.06,1.20,58.89,-0.11
3,4,148254,3.81,1.14,58.98,-0.11
4,5,147946,9.63,1.16,58.09,-0.11


momentum_quintile — The group of stocks, from weakest past performance (1) to strongest (5).

observations — Number of stock-date records in that group.

avg_past_return_pct — Average price change over the previous 20 trading sessions.

avg_future_return_pct — Average price change over the next 20 trading sessions.

positive_return_pct — Percentage of records where the future return was positive.

top_vs_bottom_spread_pct — Difference between the future returns of Group 5 and Group 1, in percentage points.